# Math and Numbers

## Quick notes
### What this covers
- Integer vs real arithmetic, the `/` and `%` operators, and what `CAST` really does.
- Scalar math functions: `round`, `abs`, `sign`, `ceil`, `floor`, `trunc`, `min` / `max` (scalar), `power`, `sqrt`, `ln`, `log`, `exp`, `mod`.
- Floating point error, why money belongs in integer cents, rounding rules (half away from zero vs banker's).
- Safe ratios and percentages, deterministic sampling, bit operations, hex literals, overflow, number formatting with `printf`, aggregates over numbers.
### Key points
- `integer / integer` is **integer division**: `7 / 2` is `3`. Make one side real (`7 / 2.0`, or `7 * 1.0 / 2`) to get `3.5`.
- `x / 0` and `x % 0` return `NULL` in SQLite. They do not raise an error.
- `round(x)` always returns a **real** (`3.0`), even with no digits argument, and rounds halves away from zero.
- Floating point cannot store `0.1` exactly. Store money as integer cents, not as REAL.
- `sum()` of integers stays an integer and **raises an error on overflow**; `total()` always returns a real and never raises.
- The math functions (`sqrt`, `power`, `ln`, `ceil`...) exist only when SQLite is compiled with `SQLITE_ENABLE_MATH_FUNCTIONS`. This one has them; the demo guards against builds that do not.
### Gotchas (the part worth revising twice)
- `CAST('12abc' AS INTEGER)` is `12`, and `CAST('abc' AS INTEGER)` is `0`. A bad cast is **silent**, it is never an error.
- `CAST(3.9 AS INTEGER)` is `3` and `CAST(-3.9 AS INTEGER)` is `-3`: it truncates toward zero, it does not round.
- `0.1 + 0.2 = 0.3` is **false**. Compare reals with a tolerance, or avoid reals.
- `%` on a real first converts both sides to integers: `7.9 % 2` gives `1.0`, not `1.9`.
- Scalar `min(a, b)` / `max(a, b)` return `NULL` if any argument is `NULL`. The aggregate `min(col)` / `max(col)` ignores `NULL`s.
- There are no binary literals (`0b101` is a syntax error). Hex literals (`0xFF`) are fine.
### When it matters
- Anything with money, percentages, averages, bucketing, IDs, flags stored as bit masks, or reports that must add up exactly.
### One-line recap example
```sql
SELECT 7 / 2, 7 / 2.0, 7 % 2, round(2.5), CAST(3.9 AS INTEGER), 1 / 0;   -- 3, 3.5, 1, 3.0, 3, NULL
```

## 01. Seed data
- A small shop: products priced two ways on purpose. `price` is REAL dollars (the risky way), `price_cents` is INTEGER cents (the safe way).
- `order_items` records quantity and a discount in percent. Some quantities are zero so we can practise safe division later.
- One product has a `NULL` weight to remind us what `NULL` does to arithmetic.

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE TABLE products (
    id          INTEGER PRIMARY KEY,
    name        TEXT NOT NULL,
    price       REAL NOT NULL,
    price_cents INTEGER NOT NULL,
    weight_g    INTEGER
);
INSERT INTO products VALUES
 (1, 'Pen',       0.10,   10,  15),
 (2, 'Notebook',  2.35,  235, 300),
 (3, 'Backpack', 49.99, 4999, 900),
 (4, 'Lamp',     19.95, 1995, NULL),
 (5, 'Cable',     0.30,   30,  25);
CREATE TABLE order_items (
    order_id     INTEGER NOT NULL,
    product_id   INTEGER NOT NULL REFERENCES products(id),
    qty          INTEGER NOT NULL,
    discount_pct INTEGER NOT NULL DEFAULT 0,
    PRIMARY KEY (order_id, product_id)
);
INSERT INTO order_items VALUES
 (100, 1, 10, 0), (100, 3, 1, 10), (101, 2, 3, 0), (101, 4, 1, 5), (102, 5, 0, 0), (102, 1, 7, 0), (103, 5, 0, 0);
SELECT * FROM products ORDER BY id;

id | name     | price | price_cents | weight_g
---+----------+-------+-------------+---------
1  | Pen      | 0.1   | 10          | 15      
2  | Notebook | 2.35  | 235         | 300     
3  | Backpack | 49.99 | 4999        | 900     
4  | Lamp     | 19.95 | 1995        | NULL    
5  | Cable    | 0.3   | 30          | 25      
(5 rows)



## 02. Integer vs real arithmetic
- If **both** sides are integers, `/` throws away the remainder: `7 / 2` is `3`, `-7 / 2` is `-3` (it truncates toward zero).
- If either side is real, you get a real: `7 / 2.0` is `3.5`.
- `typeof()` shows which you got. Integer literals like `5` stay integers; `5.0` is real.
- To force real arithmetic on integer columns, multiply by `1.0` **before** dividing: `a * 1.0 / b`. Writing `a / b * 1.0` is too late.
- A quick intro to this is also in 01-Basics/02-Filtering-And-Operators.

In [3]:
%%sql
SELECT 7 / 2          AS int_div,
       -7 / 2         AS neg_int_div,
       7 / 2.0        AS real_div,
       7 * 1.0 / 2    AS cast_by_multiply,
       7 / 2 * 1.0    AS too_late,
       typeof(7 / 2)  AS t_int,
       typeof(7 / 2.0) AS t_real;

int_div | neg_int_div | real_div | cast_by_multiply | too_late | t_int   | t_real
--------+-------------+----------+------------------+----------+---------+-------
3       | -3          | 3.5      | 3.5              | 3.0      | integer | real  
(1 row)



## 03. The % operator (modulo)
- `a % b` is the remainder. The sign follows the **left** side: `-7 % 3` is `-1`, `7 % -3` is `1`.
- `%` works on integers. Real operands are converted to integers first, so `7.9 % 2` is `1.0`.
- `a % 0` is `NULL`.
- Common uses: even / odd (`n % 2 = 0`), "every 5th row", wrapping around a cycle, extracting a digit (`n % 10`).
- For a remainder that keeps the fraction, use the math function `mod(x, y)` (section 06): `mod(7.5, 2)` is `1.5`.

In [4]:
%%sql
SELECT 7 % 3      AS seven_mod_3,
       -7 % 3     AS neg_seven_mod_3,
       7 % -3     AS seven_mod_neg_3,
       7.9 % 2    AS real_truncated_first,
       7 % 0      AS mod_zero,
       1234 % 10  AS last_digit,
       1234 / 10 % 10 AS tens_digit;

seven_mod_3 | neg_seven_mod_3 | seven_mod_neg_3 | real_truncated_first | mod_zero | last_digit | tens_digit
------------+-----------------+-----------------+----------------------+----------+------------+-----------
1           | -1              | 1               | 1.0                  | NULL     | 4          | 3         
(1 row)



## 04. round, abs, sign
- `round(x)` rounds to a whole number but returns a **real**: `round(2.5)` is `3.0`. `round(x, n)` keeps `n` decimals. A negative `n` does **not** round to tens or hundreds as in some databases: SQLite treats it as 0 (`round(1234.5678, -2)` is `1235.0`). Use `round(x / 100) * 100` to round to hundreds.
- Halves round **away from zero**: `round(2.5)` is `3.0`, `round(-2.5)` is `-3.0`.
- `abs(x)` is the distance from zero. `sign(x)` is `-1`, `0` or `1` (and `NULL` for `NULL`).
- To get an integer back, wrap it: `CAST(round(x) AS INTEGER)`.

In [5]:
%%sql
SELECT round(2.5)            AS r_half_up,
       round(-2.5)           AS r_half_down,
       round(1234.5678, 2)   AS two_places,
       round(1234.5678, -2)  AS neg_places,
       round(1234.5678 / 100) * 100 AS to_hundreds,
       CAST(round(2.5) AS INTEGER) AS as_int,
       abs(-7)               AS abs_neg,
       sign(-12)             AS sign_neg,
       sign(0)               AS sign_zero;

r_half_up | r_half_down | two_places | neg_places | to_hundreds | as_int | abs_neg | sign_neg | sign_zero
----------+-------------+------------+------------+-------------+--------+---------+----------+----------
3.0       | -3.0        | 1234.57    | 1235.0     | 1200.0      | 3      | 7       | -1       | 0        
(1 row)



## 05. Scalar min / max versus aggregate min / max
- With **two or more arguments**, `min(a, b, c)` and `max(a, b, c)` are scalar functions: they compare values in one row.
- With **one argument** they are aggregates over the rows of a group.
- Scalar versions return `NULL` if any argument is `NULL`. Aggregates skip `NULL`s.
- Scalar `max(x, 0)` is the classic "clamp at zero". Combine with `min` to clamp into a range: `max(0, min(100, x))`.

In [6]:
%%sql
SELECT min(3, 7, 5)           AS smallest,
       max(3, 7, 5)           AS largest,
       min(3, NULL, 5)        AS null_poisons,
       max(0, -12)            AS clamp_floor,
       max(0, min(100, 140))  AS clamp_range;
SELECT product_id, qty, discount_pct, max(0, qty - 3) AS qty_over_three
FROM order_items
ORDER BY order_id, product_id;
SELECT min(weight_g) AS aggregate_min, max(weight_g) AS aggregate_max FROM products;

smallest | largest | null_poisons | clamp_floor | clamp_range
---------+---------+--------------+-------------+------------
3        | 7       | NULL         | 0           | 100        
(1 row)

product_id | qty | discount_pct | qty_over_three
-----------+-----+--------------+---------------
1          | 10  | 0            | 7             
3          | 1   | 10           | 0             
2          | 3   | 0            | 0             
4          | 1   | 5            | 0             
1          | 7   | 0            | 4             
5          | 0   | 0            | 0             
5          | 0   | 0            | 0             
(7 rows)

aggregate_min | aggregate_max
--------------+--------------
15            | 900          
(1 row)



## 06. ceil, floor, trunc, and the optional math functions
- `ceil(x)` rounds up, `floor(x)` rounds down, `trunc(x)` drops the fraction (toward zero). They return reals.
- These and `sqrt`, `power` / `pow`, `exp`, `ln`, `log10`, `log2`, `mod`, `pi`, `sin`... are the **math functions**, available only if SQLite was compiled with `SQLITE_ENABLE_MATH_FUNCTIONS`. Official builds and the one Python ships usually have it.
- On a build without them you get `no such function`. The next cell is run in "expect error" mode, so the notebook works either way.
- Negative numbers show the difference: `ceil(-1.5)` is `-1.0`, `floor(-1.5)` is `-2.0`, `trunc(-1.5)` is `-1.0`.
- Out-of-domain math returns `NULL`, not an error: `sqrt(-1)` and `ln(0)` are `NULL`.

In [7]:
%%sql expect_error
SELECT ceil(1.2)   AS ceil_pos,  floor(1.8)   AS floor_pos,  trunc(1.8)   AS trunc_pos,
       ceil(-1.5)  AS ceil_neg,  floor(-1.5)  AS floor_neg,  trunc(-1.5)  AS trunc_neg;

ceil_pos | floor_pos | trunc_pos | ceil_neg | floor_neg | trunc_neg
---------+-----------+-----------+----------+-----------+----------
2.0      | 1.0       | 1.0       | -1.0     | -2.0      | -1.0     
(1 row)



In [8]:
%%sql expect_error
SELECT sqrt(16)          AS sqrt_16,
       power(2, 10)      AS two_to_10,
       power(2, 0.5)     AS sqrt_via_power,
       exp(1)            AS e,
       ln(exp(2))        AS ln_back,
       log10(1000)       AS log10_1000,
       log2(8)           AS log2_8,
       mod(7.5, 2)       AS real_mod,
       sqrt(-1)          AS sqrt_negative,
       ln(0)             AS ln_zero;

sqrt_16 | two_to_10 | sqrt_via_power     | e                 | ln_back | log10_1000 | log2_8 | real_mod | sqrt_negative | ln_zero
--------+-----------+--------------------+-------------------+---------+------------+--------+----------+---------------+--------
4.0     | 1024.0    | 1.4142135623730951 | 2.718281828459045 | 2.0     | 3.0        | 3.0    | 1.5      | NULL          | NULL   
(1 row)



## 07. Common calculations with these functions
- Distance between two points: `sqrt(dx*dx + dy*dy)`.
- Compound growth: `principal * power(1 + rate, years)`.
- Rounding up to whole units ("how many boxes of 12 do I need for n items?"): integer trick `(n + 11) / 12`, which needs no math functions and stays an integer.
- Percent change: `(new - old) * 100.0 / old`. Guard `old = 0` (section 12).

In [9]:
%%sql
SELECT sqrt((6 - 2) * (6 - 2) + (8 - 5) * (8 - 5))     AS distance,
       round(1000 * power(1 + 0.05, 3), 2)             AS grows_to,
       (25 + 11) / 12                                  AS boxes_needed,
       round((120 - 100) * 100.0 / 100, 1)             AS pct_change;

distance | grows_to | boxes_needed | pct_change
---------+----------+--------------+-----------
5.0      | 1157.63  | 3            | 20.0      
(1 row)



## 08. CAST pitfalls
- `CAST(x AS INTEGER)` truncates reals toward zero and reads the **leading digits** of text, ignoring the rest. No leading digits means `0`.
- `CAST('1e3' AS INTEGER)` is `1` (it stops at the `e`) but `CAST('1e3' AS REAL)` is `1000.0`.
- Hex text is not parsed: `CAST('0x1A' AS INTEGER)` is `0`.
- Huge reals saturate at the largest integer instead of failing.
- Because bad casts never fail, validate text first (for example `x GLOB '*[^0-9]*'` finds non-digit characters) before converting imported data.
- Text vs number comparison depends on column affinity; see 01-Basics/01-Fundamentals.

In [10]:
%%sql
SELECT CAST('12abc' AS INTEGER)  AS leading_digits,
       CAST('abc' AS INTEGER)    AS no_digits,
       CAST(' 7 ' AS INTEGER)    AS spaces_ok,
       CAST(3.9 AS INTEGER)      AS trunc_pos,
       CAST(-3.9 AS INTEGER)     AS trunc_neg,
       CAST('1e3' AS INTEGER)    AS exp_as_int,
       CAST('1e3' AS REAL)       AS exp_as_real,
       CAST('0x1A' AS INTEGER)   AS hex_text,
       CAST(1e30 AS INTEGER)     AS saturates;
SELECT '10' = 10 AS text_vs_int_literal,
       '10' < '9' AS text_compare,
       10 < '9'   AS int_vs_text,
       typeof(1 + '2') AS plus_makes_number,
       '3abc' + 1 AS partial_text_in_math;

leading_digits | no_digits | spaces_ok | trunc_pos | trunc_neg | exp_as_int | exp_as_real | hex_text | saturates          
---------------+-----------+-----------+-----------+-----------+------------+-------------+----------+--------------------
12             | 0         | 7         | 3         | -3        | 1          | 1000.0      | 0        | 9223372036854775807
(1 row)

text_vs_int_literal | text_compare | int_vs_text | plus_makes_number | partial_text_in_math
--------------------+--------------+-------------+-------------------+---------------------
0                   | 1            | 1           | integer           | 4                   
(1 row)



## 09. Floating point: why 0.1 + 0.2 is not 0.3
- A REAL is a binary floating point number (an 8-byte IEEE 754 double). `0.1`, `0.2` and `0.3` have no exact binary form, so tiny errors appear.
- `0.1 + 0.2` is `0.30000000000000004`, so `= 0.3` is false. SQLite prints about 15 significant digits, which can hide the error until it bites in an `=` or a `GROUP BY`.
- Summing ten `0.1`s gives `0.9999999999999999`.
- Fixes: compare with a tolerance (`abs(a - b) < 0.000001`), `round()` before comparing, or store integers.
- Python prints the full value (`0.30000000000000004`); the sqlite3 command-line shell shows only about 15 digits and would print `0.3`. The row `tiny_error` below shows the difference directly. Note `printf('%.20f', 0.1)` cannot reveal the error: SQLite's `printf` stops after about 16 significant digits.

In [11]:
%%sql
SELECT 0.1 + 0.2                 AS sum_value,
       0.1 + 0.2 = 0.3           AS equal_to_point_three,
       abs((0.1 + 0.2) - 0.3) < 0.000001 AS equal_with_tolerance,
       round(0.1 + 0.2, 2) = 0.3 AS equal_after_round,
       (0.1 + 0.2) - 0.3         AS tiny_error,
       printf('%.20f', 0.1)      AS printf_hides_it;
WITH RECURSIVE n(i, total) AS (
    SELECT 1, 0.1 UNION ALL SELECT i + 1, total + 0.1 FROM n WHERE i < 10
)
SELECT total, total = 1.0 AS equals_one, round(total, 9) = 1.0 AS rounded_equals_one
FROM n WHERE i = 10;

sum_value           | equal_to_point_three | equal_with_tolerance | equal_after_round | tiny_error            | printf_hides_it       
--------------------+----------------------+----------------------+-------------------+-----------------------+-----------------------
0.30000000000000004 | 0                    | 1                    | 1                 | 5.551115123125783e-17 | 0.10000000000000000000
(1 row)

total              | equals_one | rounded_equals_one
-------------------+------------+-------------------
0.9999999999999999 | 0          | 1                 
(1 row)



## 10. Money: REAL dollars (bad) vs INTEGER cents (good)
- With REAL prices, single lines can be off by a hair: Notebook, 3 x 2.35, is `7.050000000000001`, so an `= 7.05` check fails. (`sum()` itself uses compensated summation, so it hides many such errors, but not all.)
- With integer cents everything is exact: add, subtract and multiply by a whole quantity without error.
- Convert only at the edge: cents to dollars for display (`printf('%d.%02d', c / 100, c % 100)` or `c / 100.0` with `%.2f`), and dollars to cents on input with `CAST(round(x * 100) AS INTEGER)`.
- Store the currency too, or a column name that says it (`price_cents`).
- Percentages and tax: compute in integers, and decide the rounding rule once (next sections).

In [12]:
%%sql
SELECT p.name,
       p.price * i.qty                          AS real_line,
       p.price_cents * i.qty                    AS cents_line,
       printf('%d.%02d', p.price_cents * i.qty / 100, p.price_cents * i.qty % 100) AS shown
FROM order_items i JOIN products p ON p.id = i.product_id
WHERE i.order_id = 100
ORDER BY p.id;
SELECT p.name, p.price * i.qty AS real_line,
       p.price * i.qty = p.price_cents * i.qty / 100.0 AS real_matches_cents
FROM order_items i JOIN products p ON p.id = i.product_id
WHERE i.order_id IN (100, 101)
ORDER BY i.order_id, p.id;
SELECT sum(p.price_cents * i.qty) AS exact_total_cents FROM order_items i JOIN products p ON p.id = i.product_id;

name     | real_line | cents_line | shown
---------+-----------+------------+------
Pen      | 1.0       | 100        | 1.00 
Backpack | 49.99     | 4999       | 49.99
(2 rows)

name     | real_line         | real_matches_cents
---------+-------------------+-------------------
Pen      | 1.0               | 1                 
Backpack | 49.99             | 1                 
Notebook | 7.050000000000001 | 0                 
Lamp     | 19.95             | 1                 
(4 rows)

exact_total_cents
-----------------
7869             
(1 row)



## 11. Rounding: half away from zero vs banker's rounding
- SQLite's `round` goes **half away from zero**: `0.5 -> 1`, `1.5 -> 2`, `2.5 -> 3`, `-2.5 -> -3`.
- **Banker's rounding** (half to even) sends exact halves to the nearest even number, which avoids a steady upward drift across many rows: `0.5 -> 0`, `1.5 -> 2`, `2.5 -> 2`, `3.5 -> 4`.
- SQLite has no banker's function. For exact halves use `2 * round(x / 2)`; other values use plain `round`.
- A float trap: `round(2.675, 2)` gives `2.67`, not `2.68`, because `2.675` is stored as slightly less than `2.675`. Rounding **integer cents** avoids it.
- Pick one rule, write it in one place (a view or your app), and use it everywhere.

In [13]:
%%sql
WITH t(x) AS (VALUES (0.5), (1.5), (2.5), (3.5), (-0.5), (-1.5), (-2.5), (2.4), (2.6))
SELECT x,
       round(x)                                                  AS half_away,
       CASE WHEN x - floor(x) = 0.5 THEN 2 * round(x / 2) ELSE round(x) END AS bankers
FROM t
ORDER BY x;
SELECT round(2.675, 2) AS float_trap, round(267.5) / 100 AS cents_first;

x    | half_away | bankers
-----+-----------+--------
-2.5 | -3.0      | -2.0   
-1.5 | -2.0      | -2.0   
-0.5 | -1.0      | 0.0    
0.5  | 1.0       | 0.0    
1.5  | 2.0       | 2.0    
2.4  | 2.0       | 2.0    
2.5  | 3.0       | 2.0    
2.6  | 3.0       | 3.0    
3.5  | 4.0       | 4.0    
(9 rows)

float_trap | cents_first
-----------+------------
2.67       | 2.68       
(1 row)



## 12. Percentages and ratios without division by zero
- In SQLite `x / 0` is `NULL` (not an error), so a bad ratio quietly turns into `NULL`. That is safe but easy to miss.
- Other databases raise an error, so write `x / NULLIF(y, 0)` anyway: it is explicit, portable, and shows intent.
- Use `coalesce(..., 0)` only when `0` is a sensible answer for "no data"; often `NULL` (unknown) is the honest one.
- Remember integer division: `part * 100.0 / total` for a percentage, or `part * 100 / total` for a truncated whole percent.
- Orders 102 and 103 have lines with qty `0`. Order 103 has **only** such a line, so its "average per stocked line" divides by zero and comes out `NULL`, and its percentage of order quantity also divides by zero.

In [14]:
%%sql
SELECT 5 / 0 AS plain_div_zero, 5 / NULLIF(0, 0) AS nullif_div_zero;
SELECT order_id, product_id, qty,
       round(100.0 * qty / (SELECT sum(qty) FROM order_items WHERE order_id = i.order_id), 1) AS pct_of_order_qty
FROM order_items i
ORDER BY order_id, product_id;
SELECT order_id,
       sum(qty)                                         AS units,
       sum(CASE WHEN qty > 0 THEN 1 ELSE 0 END)         AS lines_with_stock,
       sum(qty) * 1.0 / NULLIF(sum(CASE WHEN qty > 0 THEN 1 ELSE 0 END), 0) AS avg_qty_per_line
FROM order_items
GROUP BY order_id
ORDER BY order_id;

plain_div_zero | nullif_div_zero
---------------+----------------
NULL           | NULL           
(1 row)

order_id | product_id | qty | pct_of_order_qty
---------+------------+-----+-----------------
100      | 1          | 10  | 90.9            
100      | 3          | 1   | 9.1             
101      | 2          | 3   | 75.0            
101      | 4          | 1   | 25.0            
102      | 1          | 7   | 100.0           
102      | 5          | 0   | 0.0             
103      | 5          | 0   | NULL            
(7 rows)

order_id | units | lines_with_stock | avg_qty_per_line
---------+-------+------------------+-----------------
100      | 11    | 2                | 5.5             
101      | 4     | 2                | 2.0             
102      | 7     | 1                | 7.0             
103      | 0     | 0                | NULL            
(4 rows)



## 13. Discounts and tax in integer math
- Work in cents. For a percent discount: `cents * (100 - pct) / 100` does integer division and **truncates**.
- To round to the nearest cent instead, add half the divisor first: `(cents * (100 - pct) + 50) / 100` (for non-negative amounts).
- Truncating each line and then summing can differ from summing then truncating by a cent or two. Decide which one your business rule says and document it.
- Here: the Backpack at 10% off and the Lamp at 5% off come out as `4499.1` and `1895.25` cents: the fraction is under half a cent, so truncating and rounding give the same whole cents and both columns agree. The second query uses a 995 cent item at 10% off, `895.5` cents, where they differ.

In [15]:
%%sql
SELECT p.name, i.qty, i.discount_pct,
       p.price_cents * i.qty                                           AS gross_cents,
       p.price_cents * i.qty * (100 - i.discount_pct) / 100            AS net_truncated,
       (p.price_cents * i.qty * (100 - i.discount_pct) + 50) / 100     AS net_rounded
FROM order_items i JOIN products p ON p.id = i.product_id
WHERE i.discount_pct > 0
ORDER BY p.id;
SELECT 995 * 90 / 100 AS truncated, (995 * 90 + 50) / 100 AS rounded;

name     | qty | discount_pct | gross_cents | net_truncated | net_rounded
---------+-----+--------------+-------------+---------------+------------
Backpack | 1   | 10           | 4999        | 4499          | 4499       
Lamp     | 1   | 5            | 1995        | 1895          | 1895       
(2 rows)

truncated | rounded
----------+--------
895       | 896    
(1 row)



## 14. Random-free sampling alternatives
- `random()` gives different rows on every run, which makes results unrepeatable (and tests flaky).
- Deterministic samples: "every Nth row" using `row_number() OVER (ORDER BY id)` and `% n`, or a hash of the key such as `(id * 2654435761) % 100 < 60` for roughly 60% (change the 60 for another share).
- `ntile(k)` splits ordered rows into `k` buckets; take bucket 1 for a simple sample. See 03-Window-Functions/01-Window-Functions.
- The same input gives the same sample every time, so a bug report can be reproduced.

In [16]:
%%sql
WITH numbered AS (
    SELECT id, name, row_number() OVER (ORDER BY id) AS rn FROM products
)
SELECT id, name, rn FROM numbered WHERE rn % 2 = 1 ORDER BY id;
SELECT id, name, (id * 2654435761) % 100 AS bucket,
       (id * 2654435761) % 100 < 60       AS in_sample
FROM products
ORDER BY id;

id | name     | rn
---+----------+---
1  | Pen      | 1 
3  | Backpack | 3 
5  | Cable    | 5 
(3 rows)

id | name     | bucket | in_sample
---+----------+--------+----------
1  | Pen      | 61     | 0        
2  | Notebook | 22     | 1        
3  | Backpack | 83     | 0        
4  | Lamp     | 44     | 1        
5  | Cable    | 5      | 1        
(5 rows)



## 15. Bit operations
- Operators: `&` and, `|` or, `~` not, `<<` shift left, `>>` shift right. They work on 64-bit signed integers.
- Use them for flags packed in one integer, like file permissions: read = 4, write = 2, execute = 1.
- Test a flag with `(mask & 2) <> 0`. Set it with `mask | 2`. Clear it with `mask & ~2`.
- `x << n` is `x * 2^n`, `x >> n` divides by `2^n` and rounds down (it keeps the sign, so `-1 >> 1` is still `-1`).
- A packed column is compact but unreadable and not indexable per flag. Prefer separate columns unless space or speed truly matters.

In [17]:
%%sql
CREATE TABLE perms (user_name TEXT PRIMARY KEY, mask INTEGER NOT NULL);
INSERT INTO perms VALUES ('ada', 7), ('alan', 5), ('grace', 4);
SELECT user_name, mask,
       (mask & 4) <> 0 AS can_read,
       (mask & 2) <> 0 AS can_write,
       (mask & 1) <> 0 AS can_exec,
       mask | 2        AS with_write,
       mask & ~1       AS without_exec
FROM perms
ORDER BY user_name;
SELECT 1 << 3 AS shift_left, 255 >> 2 AS shift_right, 6 & 3 AS b_and, 6 | 3 AS b_or, ~6 AS b_not, -1 >> 1 AS keeps_sign;

user_name | mask | can_read | can_write | can_exec | with_write | without_exec
----------+------+----------+-----------+----------+------------+-------------
ada       | 7    | 1        | 1         | 1        | 7          | 6           
alan      | 5    | 1        | 0         | 1        | 7          | 4           
grace     | 4    | 1        | 0         | 0        | 6          | 4           
(3 rows)

shift_left | shift_right | b_and | b_or | b_not | keeps_sign
-----------+-------------+-------+------+-------+-----------
8          | 63          | 2     | 7    | -7    | -1        
(1 row)



## 16. Hex literals (and no binary literals)
- Write hex integers as `0xFF`; they are normal integers (255).
- Binary literals like `0b1010` are **not** supported. Use `10`, `0xA`, or build them from shifts.
- `hex(x)` goes the other way, but note it converts the value to text first. For an integer, `hex(255)` is `'323535'` (the hex of the text `'255'`), not `'FF'`! Use `printf('%X', 255)` instead.
- `x'4142'` is a **BLOB** literal, not a number.

In [18]:
%%sql
SELECT 0xFF AS ff, 0x10 + 1 AS sixteen_plus_one, printf('%X', 255) AS via_printf, hex(255) AS hex_of_text, typeof(x'4142') AS blob_literal;

ff  | sixteen_plus_one | via_printf | hex_of_text | blob_literal
----+------------------+------------+-------------+-------------
255 | 17               | FF         | 323535      | blob        
(1 row)



In [19]:
%%sql expect_error
SELECT 0b1010 AS binary_literal;

ERROR: unrecognized token: "0b1010"



## 17. Integer limits and overflow
- Integers are signed 64-bit: from -9223372036854775808 to 9223372036854775807.
- Arithmetic with `+ - *` that overflows silently becomes a **real**: `9223372036854775807 + 1` is `9.223372036854776e+18`, and precision is lost.
- `sum()` of integers is stricter: it raises `integer overflow`. Use `total()` if you want a real result and no error.
- `abs(-9223372036854775808)` also raises `integer overflow`.
- If your numbers can get this big (not money in cents, but maybe counters in a log), switch to reals knowingly or split into parts.

In [20]:
%%sql
SELECT 9223372036854775807 + 1                 AS overflow_to_real,
       typeof(9223372036854775807 + 1)         AS t,
       typeof(9223372036854775807)             AS t_max,
       9223372036854775807 * 2                 AS big_mult,
       1e308 * 10                              AS real_overflow;

overflow_to_real      | t    | t_max   | big_mult               | real_overflow
----------------------+------+---------+------------------------+--------------
9.223372036854776e+18 | real | integer | 1.8446744073709552e+19 | inf          
(1 row)



In [21]:
%%sql expect_error
SELECT sum(x) FROM (SELECT 9223372036854775807 AS x UNION ALL SELECT 1);

ERROR: integer overflow



In [22]:
%%sql
SELECT total(x) AS total_is_real FROM (SELECT 9223372036854775807 AS x UNION ALL SELECT 1);

total_is_real        
---------------------
9.223372036854776e+18
(1 row)



In [23]:
%%sql expect_error
SELECT abs(-9223372036854775808);

ERROR: integer overflow



## 18. Number formatting with printf
- `%d` integer, `%f` real (default 6 decimals), `%.2f` two decimals, `%5.1f` width 5 with one decimal, `%08.3f` zero padded, `%+d` always show the sign, `%x` / `%X` hex, `%o` octal, `%e` scientific.
- A comma flag (`%,d`) puts thousands separators. This works in recent SQLite versions but is not standard, so test it where you deploy.
- Formatting returns **text**. Do not sum it or sort by it as numbers; format at the very last step.
- Rounding in `%.2f` uses the real's exact stored value, so `printf('%.2f', 2.675)` also gives `2.67` (section 11).

In [24]:
%%sql
SELECT printf('%d', 42)              AS d,
       printf('%.2f', 3.14159)       AS two_dec,
       printf('%08.3f', 3.14159)     AS zero_padded,
       printf('%+d', 5)              AS signed,
       printf('%X', 255)             AS hex_upper,
       printf('%o', 8)               AS octal,
       printf('%e', 12345.678)       AS scientific,
       printf('%,d', 1234567)        AS thousands,
       printf('%5.1f%%', 45.678)     AS percent,
       printf('%.2f', 2.675)         AS exact_value_rounds,
       typeof(printf('%d', 42))      AS result_type;

d  | two_dec | zero_padded | signed | hex_upper | octal | scientific   | thousands | percent | exact_value_rounds | result_type
---+---------+-------------+--------+-----------+-------+--------------+-----------+---------+--------------------+------------
42 | 3.14    | 0003.142    | +5     | FF        | 10    | 1.234568e+04 | 1,234,567 |  45.7%  | 2.67               | text       
(1 row)



## 19. Aggregates over numbers: sum, avg, total, count
- `sum` of integers is an integer; of anything with a real in it, a real. `total` is always real.
- `avg` is always real: the average of the values 1 and 2 is `1.5`. A hand-made `sum(x) / count(x)` over integers is **integer division** and loses the fraction.
- All aggregates skip `NULL`s. `avg` divides by the count of non-`NULL` values, so it differs from "total over all rows".
- On zero rows: `sum` and `avg` and `max` are `NULL`, `count` is `0`, `total` is `0.0`.
- Text that looks like a number is converted in `sum`; text that does not counts as `0`.
- More on grouping in 02-Querying/03-Aggregation.

In [25]:
%%sql
SELECT count(*)         AS rows_,
       count(weight_g)  AS non_null_weights,
       sum(weight_g)    AS sum_w,
       avg(weight_g)    AS avg_w,
       sum(weight_g) / count(weight_g) AS manual_int_avg,
       sum(price_cents) / count(*)     AS int_avg_cents,
       avg(price_cents)                AS real_avg_cents,
       sum(weight_g) * 1.0 / count(*)  AS avg_if_null_is_zero,
       total(weight_g)  AS total_w,
       typeof(sum(weight_g)) AS t_sum,
       typeof(total(weight_g)) AS t_total
FROM products;
SELECT sum(weight_g) AS sum_empty, total(weight_g) AS total_empty, avg(weight_g) AS avg_empty, count(weight_g) AS count_empty, max(weight_g) AS max_empty
FROM products WHERE id < 0;
SELECT sum(x) AS sum_mixed_text FROM (SELECT '5' AS x UNION ALL SELECT '2.5' UNION ALL SELECT 'abc');

rows_ | non_null_weights | sum_w | avg_w | manual_int_avg | int_avg_cents | real_avg_cents | avg_if_null_is_zero | total_w | t_sum   | t_total
------+------------------+-------+-------+----------------+---------------+----------------+---------------------+---------+---------+--------
5     | 4                | 1240  | 310.0 | 310            | 1453          | 1453.8         | 248.0               | 1240.0  | integer | real   
(1 row)

sum_empty | total_empty | avg_empty | count_empty | max_empty
----------+-------------+-----------+-------------+----------
NULL      | 0.0         | NULL      | 0           | NULL     
(1 row)

sum_mixed_text
--------------
7.5           
(1 row)



## 20. Putting it together: an invoice in exact cents
- Line total = price * qty, discount applied in integer math with round-to-nearest, tax added at the end.
- All amounts stay integers until the final `printf`. That keeps the report exact and totals consistent with the lines.
- Tax here is 8% on the discounted subtotal, rounded to the nearest cent.

In [26]:
%%sql
WITH lines AS (
    SELECT i.order_id, p.name, i.qty,
           (p.price_cents * i.qty * (100 - i.discount_pct) + 50) / 100 AS net_cents
    FROM order_items i JOIN products p ON p.id = i.product_id
),
orders AS (
    SELECT order_id, sum(net_cents) AS subtotal FROM lines GROUP BY order_id
)
SELECT order_id,
       printf('%,.2f', subtotal / 100.0)                  AS subtotal,
       printf('%.2f', (subtotal * 8 + 50) / 100 / 100.0)  AS tax_8pct,
       printf('%.2f', (subtotal + (subtotal * 8 + 50) / 100) / 100.0) AS total
FROM orders
ORDER BY order_id;

order_id | subtotal | tax_8pct | total
---------+----------+----------+------
100      | 45.99    | 3.68     | 49.67
101      | 26.00    | 2.08     | 28.08
102      | 0.70     | 0.06     | 0.76 
103      | 0.00     | 0.00     | 0.00 
(4 rows)

